## Two Flavour oscillation
- This is more likely a learning path, before jumping into Three flavour oscillation curve.
- The standard two flavour oscillation probability is given by the formula: P(να​→νβ​)= sin^2(2θ)sin^2(1.27(Δm^2 L)/E​)
- This shows the probability that a muon neutrino beomes an electron neutrino.
- There are various formulas that provides appreance, disappereance probabilty for muon, electron neutrino.
- In this notebook we will learn about the electron neutrino appearance (P(νμ → νe)), which can be learned from the formula above and we will be learning about the Muon neutrino disappearance(P(νμ → νμ))

The electron neutrino appearance probability is

$$
P(\nu_\mu \rightarrow \nu_e)
=
\sin^2(2\theta)\,
\sin^2\left(1.27\frac{\Delta m^2 L}{E}\right)
$$

The muon neutrino survival probability is

$$
P(\nu_\mu \rightarrow \nu_\mu)
=
1 -
\sin^2(2\theta)\,
\sin^2\left(1.27\frac{\Delta m^2 L}{E}\right)
$$

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("widget")
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display


N_POINTS = 500


def appearance_probability(L_km, E_GeV, dm2, theta):
    """Two-flavour appearance probability P(ν_mu -> ν_e)."""
    L_km = np.asarray(L_km, dtype=float)
    E_GeV = np.asarray(E_GeV, dtype=float)
    phase = 1.267 * dm2 * L_km / E_GeV
    return np.sin(2.0 * theta) ** 2 * np.sin(phase) ** 2


def survival_probability(L_km, E_GeV, dm2, theta):
    """Two-flavour survival probability P(ν_mu -> ν_mu)."""
    return 1.0 - appearance_probability(L_km, E_GeV, dm2, theta)


scan_mode = w.ToggleButtons(
    options=[("Energy scan", "energy"), ("Hypothetical baseline scan", "length")],
    value="energy",
    description="Scan mode:",
    style={"description_width": "initial"},
)

w_L = w.FloatSlider(
    value=295.0,
    min=1.0,
    max=3000.0,
    step=1.0,
    description="Baseline L (km)",
    style={"description_width": "110px"},
    layout=w.Layout(width="420px"),
)
w_E = w.FloatSlider(
    value=0.60,
    min=0.05,
    max=20.0,
    step=0.01,
    description="Neutrino E (GeV)",
    style={"description_width": "110px"},
    layout=w.Layout(width="420px"),
)
w_Lmax = w.FloatSlider(
    value=1500.0,
    min=50.0,
    max=5000.0,
    step=50.0,
    description="Baseline max (km)",
    style={"description_width": "130px"},
    layout=w.Layout(width="420px"),
)
w_Emax = w.FloatSlider(
    value=10.0,
    min=1.0,
    max=30.0,
    step=0.5,
    description="Energy max (GeV)",
    style={"description_width": "130px"},
    layout=w.Layout(width="420px"),
)
w_dm2 = w.FloatSlider(
    value=2.5e-3,
    min=1.0e-3,
    max=4.0e-3,
    step=1.0e-5,
    description="Δm² (eV²)",
    readout_format=".5f",
    style={"description_width": "110px"},
    layout=w.Layout(width="420px"),
)
w_theta = w.FloatSlider(
    value=33.0,
    min=0.0,
    max=90.0,
    step=0.1,
    description="θ (deg)",
    style={"description_width": "110px"},
    layout=w.Layout(width="420px"),
)

fig, axes = plt.subplot_mosaic(
    [["appearance", "survival"], ["overview", "overview"]],
    figsize=(16, 8),
    constrained_layout=True,
)
out = w.Output()


def _draw_curves(*_):
    theta = np.radians(w_theta.value)
    dm2 = w_dm2.value

    if scan_mode.value == "energy":
        x = np.linspace(0.05, w_Emax.value, N_POINTS)
        fixed_value = w_L.value
        x_label = "Energy E (GeV)"
        scan_text = f"L = {fixed_value:.0f} km"
        scan_title = "Energy scan"
        appearance = appearance_probability(fixed_value, x, dm2, theta)
    else:
        x = np.linspace(0.0, w_Lmax.value, N_POINTS)
        fixed_value = w_E.value
        x_label = "Baseline L (km)"
        scan_text = f"E = {fixed_value:.2f} GeV"
        scan_title = "Hypothetical baseline scan"
        appearance = appearance_probability(x, fixed_value, dm2, theta)

    survival = 1.0 - appearance

    for ax in axes.values():
        ax.cla()
        ax.grid(alpha=0.25)
        ax.set_ylim(-0.02, 1.05)
        ax.set_xlabel(x_label)
        ax.set_ylabel("Probability")

    axes["appearance"].plot(x, appearance, color="#e25c2a", lw=2.2, label=r"P($\nu_\mu \rightarrow \nu_e$)")
    axes["appearance"].set_title(f"Appearance - {scan_title}\n{scan_text}")
    axes["appearance"].legend(fontsize=9)

    axes["survival"].plot(x, survival, color="#3a7abf", lw=2.2, label=r"P($\nu_\mu \rightarrow \nu_\mu$)")
    axes["survival"].set_title(f"Survival - {scan_title}\n{scan_text}")
    axes["survival"].legend(fontsize=9)

    axes["overview"].plot(x, appearance, color="#e25c2a", lw=2.0, label=r"P($\nu_\mu \rightarrow \nu_e$)")
    axes["overview"].plot(x, survival, color="#3a7abf", lw=2.0, label=r"P($\nu_\mu \rightarrow \nu_\mu$)")
    axes["overview"].set_title(f"Overview - {scan_title}\n{scan_text}")
    axes["overview"].legend(fontsize=9)

    if scan_mode.value == "energy":
        x_max = w_Emax.value
    else:
        x_max = w_Lmax.value
    for ax in axes.values():
        ax.set_xlim(x.min(), x_max)

    fig.suptitle(
        rf"Two-flavour oscillation: $\sin^2(2\theta)$ $\sin^2\!\left(1.267\,\Delta m^2\,L/E\right)$",
        fontsize=13,
    )
    fig.canvas.draw_idle()


def _update_controls(*_):
    """Update visible controls based on scan mode."""
    if scan_mode.value == "energy":
        controls.children = [
            w.HTML("<b>Two-flavour oscillation explorer</b>"),
            w.HTML("Energy scan: fixed baseline, varying energy."),
            w.HBox([scan_mode]),
            w.HBox([w_L]),
            w.HBox([w_Emax]),
            w.HBox([w_dm2, w_theta]),
        ]
    else:
        controls.children = [
            w.HTML("<b>Two-flavour oscillation explorer</b>"),
            w.HTML("Hypothetical baseline scan: fixed energy, varying baseline."),
            w.HBox([scan_mode]),
            w.HBox([w_E]),
            w.HBox([w_Lmax]),
            w.HBox([w_dm2, w_theta]),
        ]


for widget in (w_L, w_E, w_Lmax, w_Emax, w_dm2, w_theta):
    widget.observe(_draw_curves, names="value")

scan_mode.observe(_draw_curves, names="value")
scan_mode.observe(_update_controls, names="value")

controls = w.VBox(
    [
        w.HTML("<b>Two-flavour oscillation explorer</b>"),
        w.HTML("Energy scan: fixed baseline, varying energy."),
        w.HBox([scan_mode]),
        w.HBox([w_L]),
        w.HBox([w_Emax]),
        w.HBox([w_dm2, w_theta]),
    ],
    layout=w.Layout(border="1px solid #ddd", padding="8px", margin="4px"),
)

with out:
    display(fig.canvas)

display(controls, out)
_draw_curves()


Output()